# OOF Error Analysis — DINOv3 LoRA 384

This notebook analyzes wrong out-of-fold predictions from `outputs_dinov3_lora_384` to identify class confusion, ambiguity, possible label noise, duplicate conflicts, image-quality problems, and fold-specific weaknesses.

It only uses training/OOF data. It does not use test labels and does not modify the dataset.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage, Markdown

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import error_analysis as ea

DATA_ROOT = REPO_ROOT / "BDC2026"
OUTPUT_DIR = REPO_ROOT / "outputs_dinov3_lora_384"
ANALYSIS_DIR = OUTPUT_DIR / "error_analysis"
COMPUTE_IMAGE_METADATA = False  # Set True for brightness/contrast/blur/size analysis.
EDA_DIRS = [REPO_ROOT / "eda_outputs", REPO_ROOT / "eda_outputs_dino"]

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
assert (OUTPUT_DIR / "oof_predictions.csv").exists(), "oof_predictions.csv not found"


## 1. Run analysis

This creates enriched predictions, confusion reports, training curves, visual grids, and CSV files in `outputs_dinov3_lora_384/error_analysis/`.


In [ ]:
result = ea.run_analysis(
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    analysis_dir=ANALYSIS_DIR,
    compute_metadata=COMPUTE_IMAGE_METADATA,
    max_metadata_images=0,
    eda_dirs=EDA_DIRS,
    high_confidence=0.85,
    very_high_confidence=0.95,
    low_margin=0.15,
)

pred = result["predictions"]
class_metrics = result["class_metrics"]
cm = result["confusion_matrix"]
pairs = result["confusion_pairs"]
categories = result["error_categories"]
history = result["history"]

display(pd.Series(result["summary"], name="value").to_frame())


## 2. Metrics, confusion matrix, and training curves

Pay special attention to Electronic recall/F1 and the largest directional confusion pairs.


In [ ]:
display(class_metrics)
display(cm)
display(pairs.head(15))

for filename in [
    "confusion_matrix.png",
    "training_macro_f1.png",
    "training_valid_loss.png",
    "training_train_loss.png",
]:
    path = ANALYSIS_DIR / filename
    if path.exists():
        display(Markdown(f"### {filename}"))
        display(DisplayImage(filename=str(path)))


## 3. High-confidence wrong predictions

These are the strongest candidates for label noise, duplicate conflicts, or systematic model shortcuts.


In [ ]:
wrong = pred[~pred["correct"]].copy()
high_conf_wrong = wrong.sort_values(["confidence", "true_probability"], ascending=[False, True])

cols = [
    "resolved_path", "fold", "true_name", "pred_name", "confidence",
    "true_probability", "margin", "normalized_entropy",
    "duplicate_sources", "is_cross_label_duplicate", "error_category",
]
display(high_conf_wrong[cols].head(40))

ea.show_image_grid(
    high_conf_wrong,
    n=25,
    cols=5,
    sort_by="confidence",
    ascending=False,
    title="Highest-confidence wrong OOF predictions",
)


## 4. Most ambiguous samples

Low margin and high entropy usually indicate genuine overlap between classes, difficult crops, or missing visual detail.


In [ ]:
ambiguous = pred.sort_values(["margin", "normalized_entropy"], ascending=[True, False])
display(ambiguous[
    ["resolved_path", "fold", "true_name", "pred_name", "confidence",
     "true_probability", "margin", "normalized_entropy", "correct", "error_category"]
].head(40))

ea.show_image_grid(
    ambiguous,
    n=25,
    cols=5,
    sort_by="margin",
    ascending=True,
    title="Most ambiguous OOF predictions",
)


## 5. Inspect a specific confusion pair

Change the two class names to inspect another directional error.


In [ ]:
TRUE_CLASS = "Electronic"
PRED_CLASS = "Recyclable"

pair_rows = pred[
    (~pred["correct"])
    & (pred["true_name"] == TRUE_CLASS)
    & (pred["pred_name"] == PRED_CLASS)
].sort_values("confidence", ascending=False)

print(f"{TRUE_CLASS} → {PRED_CLASS}: {len(pair_rows)} errors")
display(pair_rows.head(30))
ea.show_image_grid(pair_rows, n=25, cols=5, title=f"{TRUE_CLASS} → {PRED_CLASS}")


## 6. Error categories and manual label review

The categories are heuristics. Never automatically relabel images without human review.

- `possible_label_noise`: very high-confidence wrong prediction with near-zero probability for the dataset label.
- `cross_label_duplicate_review`: duplicate evidence with conflicting labels.
- `duplicate_or_near_duplicate`: exact, pHash, or DINO duplicate evidence.
- `low_visual_quality`: unusual size, aspect ratio, brightness, contrast, or sharpness.
- `intrinsic_ambiguity`: low margin or high entropy.
- `systematic_model_confusion`: confident error without an obvious data-quality flag.


In [ ]:
display(categories)

review_candidates = pred[
    pred["error_category"].isin(["possible_label_noise", "cross_label_duplicate_review"])
].sort_values("confidence", ascending=False)

display(review_candidates[
    ["resolved_path", "fold", "true_name", "pred_name", "confidence",
     "true_probability", "margin", "duplicate_sources",
     "is_cross_label_duplicate", "error_category"]
].head(50))

ea.show_image_grid(review_candidates, n=25, cols=5, title="Manual label-review candidates")


## 7. Image-quality analysis

Set `COMPUTE_IMAGE_METADATA = True` in the configuration cell and rerun the notebook. This reads all training images and can take several minutes.


In [ ]:
quality_features = ["min_side", "aspect_ratio", "file_size_bytes", "brightness", "contrast", "sharpness"]
available = [c for c in quality_features if c in pred.columns and pred[c].notna().any()]
print("Available quality features:", available)

for feature in available:
    summary = ea.quality_error_summary(pred, feature, bins=5)
    if summary.empty:
        continue
    display(Markdown(f"### Error rate by `{feature}`"))
    display(summary)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(summary["bin"].astype(str), summary["error_rate"])
    ax.set_title(f"OOF error rate by {feature}")
    ax.set_ylabel("Error rate")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    plt.show()


## 8. Fold diagnostics

A consistently weak fold can indicate distribution shift, unresolved duplicate groups, or difficult images concentrated in that split.


In [ ]:
if history.empty:
    print("No fold history CSV files found.")
else:
    fold_summary = history.groupby("fold", as_index=False).agg(
        best_macro_f1=("macro_f1", "max"),
        min_valid_loss=("valid_loss", "min"),
        final_macro_f1=("macro_f1", "last"),
        epochs_trained=("epoch", "max"),
    ).sort_values("best_macro_f1")
    display(fold_summary)

error_by_fold = pred.groupby("fold", as_index=False).agg(
    images=("path", "size"),
    errors=("correct", lambda x: int((~x).sum())),
    error_rate=("correct", lambda x: 1.0 - x.mean()),
    mean_confidence=("confidence", "mean"),
).sort_values("error_rate", ascending=False)
display(error_by_fold)


## 9. Optional DINO embedding PCA

If DINO embeddings from EDA exist, this map can reveal clusters with many wrong predictions or suspected mislabeled subgroups.


In [ ]:
from sklearn.decomposition import PCA

embedding_dir = REPO_ROOT / "eda_outputs_dino"
embedding_path = embedding_dir / "dino_embeddings.npy"
index_path = embedding_dir / "dino_embedding_index.csv"

if embedding_path.exists() and index_path.exists():
    embeddings = np.load(embedding_path)
    embedding_index = pd.read_csv(index_path)
    if len(embeddings) != len(embedding_index):
        print("Embedding count does not match index count; skipping PCA.")
    else:
        embedding_index["key"] = embedding_index["class_name"].astype(str) + "/" + embedding_index["path"].map(lambda p: Path(str(p)).name)
        pred_map = pred.copy()
        pred_map["key"] = pred_map["class_name"].astype(str) + "/" + pred_map["resolved_path"].map(lambda p: Path(str(p)).name)
        lookup = pred_map.set_index("key")[["correct", "true_name", "pred_name", "confidence", "error_category"]]
        merged = embedding_index.join(lookup, on="key")
        valid = merged["correct"].notna().to_numpy()
        coords = PCA(n_components=2, random_state=42).fit_transform(embeddings)
        correct_mask = valid & merged["correct"].fillna(False).to_numpy(dtype=bool)
        wrong_mask = valid & ~merged["correct"].fillna(True).to_numpy(dtype=bool)
        fig, ax = plt.subplots(figsize=(9, 7))
        ax.scatter(coords[correct_mask, 0], coords[correct_mask, 1], s=8, alpha=0.25, label="correct")
        ax.scatter(coords[wrong_mask, 0], coords[wrong_mask, 1], s=18, alpha=0.75, label="wrong")
        ax.set_title("DINO embedding PCA: correct vs wrong OOF predictions")
        ax.legend()
        fig.tight_layout()
        plt.show()
else:
    print("DINO embeddings not found. Run:")
    print("python scripts/eda_cleaning.py --data-root ./BDC2026 --output-dir ./eda_outputs_dino --use-dino-duplicates")


## 10. Exported reports and recommended workflow

Reports are written to `outputs_dinov3_lora_384/error_analysis/`:

- `misclassified_all.csv`
- `high_confidence_wrong.csv`
- `most_uncertain_predictions.csv`
- `label_review_candidates.csv`
- `per_class_metrics.csv`
- `confusion_matrix.csv`
- `confusion_pairs.csv`
- `error_category_summary.csv`
- `error_analysis_summary.md`
- visual PNG reports

Recommended order:

1. Review high-confidence wrong samples.
2. Review cross-label duplicate candidates.
3. Inspect the largest directional confusion pair.
4. Compare errors across folds and image-quality bins.
5. Only clean or relabel after manual verification.
6. Retrain with the same folds and compare OOF Macro-F1.


In [ ]:
print("Analysis directory:", ANALYSIS_DIR.resolve())
for path in sorted(ANALYSIS_DIR.glob("*")):
    print("-", path.name)
